# Gold Processing — Product

Enriches `ecommerce.silver.silver_products` with descriptive attributes into `ecommerce.gold.gold_product`:

* **`category_name`** pulled from `ecommerce.silver.silver_category` (join on `category_code`)
* **`brand_name`** pulled from `ecommerce.silver.silver_brands` (join on `brand_code`)
* Names placed beside their codes: `category_code`, `category_name`, `brand_code`, `brand_name`
* All other product columns and metadata (`source_file`, `ingested_at`) pass through
* **Cell 3 (the table write) is only run after explicit approval** — Cell 2 shows a preview first

In [0]:
from pyspark.sql.functions import col

# Build the enriched gold DataFrame (preview only — nothing is written yet)
df_gold_product = (
    spark.table("ecommerce.silver.silver_products")
    .join(
        spark.table("ecommerce.silver.silver_category").select("category_code", "category_name"),
        on="category_code",
        how="left",
    )
    .join(
        spark.table("ecommerce.silver.silver_brands").select("brand_code", "brand_name"),
        on="brand_code",
        how="left",
    )
    # Names beside their codes, then the rest of the product columns
    .select(
        "product_id",
        "sku",
        "category_code",
        "category_name",
        "brand_code",
        "brand_name",
        "color",
        "size",
        "material",
        "weight_grams",
        "length_cm",
        "width_cm",
        "height_cm",
        "rating_count",
        "source_file",
        "ingested_at",
    )
)

print(f"Rows: {df_gold_product.count()}")
print(
    "Unmatched categories:",
    df_gold_product.where(col("category_name").isNull()).count(),
    "| Unmatched brands:",
    df_gold_product.where(col("brand_name").isNull()).count(),
)
display(df_gold_product.limit(10))

In [0]:
# Write to ecommerce.gold.gold_product — run only after approval
target_table = "ecommerce.gold.gold_product"

(
    df_gold_product.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
# Verify the persisted gold table
spark.sql("""
    SELECT
      COUNT(*) AS rows,
      COUNT_IF(category_name IS NULL) AS null_category_names,
      COUNT_IF(brand_name IS NULL) AS null_brand_names,
      COUNT(DISTINCT category_name) AS distinct_categories,
      COUNT(DISTINCT brand_name) AS distinct_brands
    FROM ecommerce.gold.gold_product
""").show()

display(spark.table("ecommerce.gold.gold_product").limit(5))